# Examine Analysis: 

Notebook that used the classes from attpc_analysis for the analysis notebooks

In [ ]:
# Ideally, you should not edit the next cell
experiment_tag = "exp" # indicate the name of the configuration file

In [ ]:
# NOTE FOR USER: You can collapse this cell
# Personal NOTE: this could be a single function, and create global variables
import yaml 
from pathlib import Path
from spyral_utils.nuclear import NuclearDataMap
from spyral_utils.plot.histogram import Histogrammer
import matplotlib.pyplot as plt 
import numpy as np
import matplotlib as mpl
cmap = mpl.colormaps["rainbow"]
%matplotlib widget

# This is for the meantime -------------------------------------------------------
# The final motivation is to make it pip installeable 
import sys
src_path = Path.cwd().parents[1]
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
# --------------------------------------------------------------------------------
from attpc_analysis.src.detector_settings.energy_loss_setup import EnergyLossSetup
from attpc_analysis.src.data_models.threebodydecay import dalitz_plot
from attpc_analysis.src.load.io import RunSolverDataLoader, RunEstimationDataLoader
from attpc_analysis.src.data_models.physics import DataReactionKinematics

nuclear_map = NuclearDataMap()

with open(f"./config/{experiment_tag}.yaml","r") as file: 
    config = yaml.safe_load(file)


# Import variables: paths, runs, experiments details, and detector settings,  
## paths to folders 
paths_to_experiment = config["paths_to_experiment"]
workspace_path = Path(paths_to_experiment["workspace_path"])
trace_path = Path(paths_to_experiment["trace_path"])
target_material_path = Path(paths_to_experiment["target_path"])

solver_result_path = workspace_path/ "InterpSolver"

## run details 
runs_details = config["runs"]
run_min = runs_details["run_min"]
run_max = runs_details["run_max"]
runs_to_skip = runs_details["runs_to_skip"]

## detector settings
detector_settings = config["detector_settings"]
ic_pressure = detector_settings["ic_pressure"]

## experiment details 
experiment_details = config["experiment_details"]
beam_energy = experiment_details ["beam_energy"]

### properties of each nuclear component 
beam_z, beam_a = experiment_details["beam"]
target_z, target_a = experiment_details["target"]
targetlike_z, targetlike_a =experiment_details["targetlike"]
beamlike_z, beamlike_a = beam_z+target_z-targetlike_z, beam_a + target_a - targetlike_a

beam = nuclear_map.get_data(beam_z, beam_a)
target = nuclear_map.get_data(target_z, target_a)
beamlike = nuclear_map.get_data(beamlike_z, beamlike_a)
targetlike = nuclear_map.get_data(targetlike_z, targetlike_a)


# Get the beam energy for the active area of the detector
energy_loss_setup = EnergyLossSetup(
    beam_energy = beam_energy,
    projectile = beam,
    attpc_target_path_json= target_material_path,
    ic_gas_pressure=600
)

target_material = energy_loss_setup.active_gas_target_material()
proj_energy_start = energy_loss_setup.beam_energy_entrance_attpc()
proj_energy_stop = energy_loss_setup.beam_energy_exit_attpc()

print(f"Reaction: {target}({beam}, {beamlike}){targetlike}")
print(f"Target material: {target_material.ugly_string}")
print(f"Accelerator Beam energy: {beam_energy} MeV")
print(f"Beam energy range in AT-TPC: {proj_energy_start}-{proj_energy_stop} MeV")

In [ ]:
# Date storage
results = {}
results["run"] = np.empty(0,int)
results["event"] = np.empty(0,int)

# Store Physics into dictations 
dictations_items = ["brho","kinetic_energy", "polar", "az",  "redchisq", "vertex_x", "vertex_y", "vertex_z","ex","polar_cm"]

for items in dictations_items:
    results[items] = np.empty(0,float)

# Load class with general details of the runs
empty_files = []
solver_data = RunSolverDataLoader(
                solver_result_path=solver_result_path, 
                particle=targetlike,
                empty_files = empty_files
                )
for run in range(run_min, run_max+1):
    df = solver_data.load_solver(run)
    if df is None: continue 
    #------------------------------------------------------------------------------
    # Insert specific filters into the dataframe 
    # (Here)
    #------------------------------------------------------------------------------
    kinematics = DataReactionKinematics(
        data = df, 
        projectile= beam, 
        target = target, 
        targetlike = targetlike, 
        beamlike = beamlike, 
        beam_energy_entrace_tpc= proj_energy_start, 
    )
    # Clear data frame
    events = df.select("event").to_numpy().flatten()
    df = None 
    
    #------------------------------------------------------------------------------
    # Do other things
    # Insert specific actions that will be saved in arrays
    #------------------------------------------------------------------------------

    # Save information 
    results["run"] = np.append(results["run"],np.full(len(events),run))
    results["event"] = np.append(results["event"], events)
    kinematics.extract_kinematics_values(results, target_material)


print("Files without data", empty_files)


## Create and visualize histograms

In [ ]:
residual_excitations = np.array([0,0.4291,4.57,6.73]) #7Be
ngrid = 1000
eject_kinematics = np.zeros((len(residual_excitations), ngrid * 2, 2))
eject_kinematics[:, :1000, :] = kinematics.kinematic_curve(proj_energy_start,residual_excitations)
eject_kinematics[:, 1000:, :] = np.flip(kinematics.kinematic_curve(proj_energy_stop, residual_excitations),axis=1)

In [ ]:
# Binning 
min_bin, max_bin = -2 , 20
bin_size = 100
granularity = (max_bin - min_bin) / bin_size


# Plot
grammer = Histogrammer()
grammer.add_hist1d("ex", bin_size, (min_bin, max_bin))
grammer.fill_hist1d("ex", results["ex"][:])
hist = grammer.get_hist1d("ex")
fig, ax = plt.subplots(1,1)
ax.stairs(hist.counts, edges=hist.bins)
ax.set_xlabel(rf"{beamlike.get_latex_rep()} Energy Spectrum (MeV)", fontsize=16)
ax.set_ylabel(f"Counts/{int(granularity*1000)} keV", fontsize=16)

colors = cmap(np.linspace(0,1,len(residual_excitations)))
for idx, ex in enumerate(residual_excitations):
    ax.axvline(ex,label=fr"$E_x$={ex:0.2f} MeV",color=colors[idx] )
ax.legend()

In [ ]:
# Make angle cuts 
make_cuts = False
show_kinematic_line = True 

angle = 10
step = 20
mask = (results["polar_cm"]>np.deg2rad(angle)) & (results["polar_cm"]<np.deg2rad(angle+step)) # lab cuts


if not make_cuts: 
    mask = np.full(len(results["ex"]), True)


plt.close("all")
fig, axs = plt.subplots(1,3,figsize=(16,6))
fig.suptitle(f"{nuclear_map.get_data(4,7).get_latex_rep()}({nuclear_map.get_data(1,2).get_latex_rep()},{nuclear_map.get_data(1,2).get_latex_rep()}){nuclear_map.get_data(4,7).get_latex_rep()}")

# Figure 1: Kinematics
ax = axs[0]
grammer.add_hist2d("kinematics", (200,200), ((0,90),(0,70)))
grammer.fill_hist2d("kinematics", np.rad2deg(results["polar"][mask]), results["kinetic_energy"][mask])
hist = grammer.get_hist2d("kinematics")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
# ax.scatter(np.rad2deg(red_points["polar"]), red_points["kinetic"],s=3, color='red')
ax.set_xlabel(r"$\theta_\text{lab}$ (deg)")
ax.set_ylabel("Kinetic energy (MeV)")

if show_kinematic_line:
    cmap = mpl.colormaps["rainbow"]
    colors = cmap(np.linspace(0,1,len(residual_excitations)))

    for idx, ex in enumerate(residual_excitations):
        ax.plot(np.rad2deg(eject_kinematics[idx, :ngrid, 0]), eject_kinematics[idx, :ngrid, 1], label=fr"$E_x$={ex:0.2f} MeV",color=colors[idx])
        ax.fill(np.rad2deg(eject_kinematics[idx, :, 0]), eject_kinematics[idx, :, 1],color=colors[idx],alpha=0.2)#, label=f"Ex={ex:02} Band", alpha=0.2)
print(np.sum(hist.counts))
ax.grid()
ax.set_xlim([0,90])
ax.legend()
# Figure 2: Ex vs Polar (cm) 
ax = axs[1]
grammer.add_hist2d("ex_polarcm",(200,200), ((0,180),(-5,20)))
grammer.fill_hist2d("ex_polarcm",np.rad2deg(results["polar_cm"][mask]), results["ex"][mask])
hist = grammer.get_hist2d("ex_polarcm")
mesh = ax.pcolormesh(hist.x_bins, hist.y_bins, hist.counts, norm='log')
# ax.set_ylim([-5,20])
# fig.colorbar(mesh)
ax.set_xlabel(r"$\theta_\text{cm}$ (deg)")
ax.set_ylabel(r"$E_x$ (MeV)")
ax.grid()

if show_kinematic_line: 
    for idx, ex in enumerate(residual_excitations):
        if ex>20: continue
        ax.axhline(ex,label=fr"$E_x$={ex:02} MeV",color=colors[idx] )


# Figure 3: Ex 
ax = axs[2]
min_bin, max_bin = -5 , 14
bin_size = 38
granularity = (max_bin - min_bin) / bin_size
# missing mass 
grammer.add_hist1d("ex", bin_size, (min_bin,max_bin))
grammer.fill_hist1d("ex", results["ex"][mask])
hist  = grammer.get_hist1d("ex")
uncertainty = np.sqrt(np.maximum(hist.counts,1))
hist_x = hist.bins[:-1] + 0.5*hist.bin_width
ax.errorbar(hist_x, hist.counts, yerr=uncertainty, fmt='o', markersize=3, capsize=2,color='black',label='missing')
# ax.scatter(hist_x, hist.counts,s=10,color='black', label='missing')
# ax.stairs(hist.counts,edges=hist.bins,color='black')

ax.set_ylabel(f"Counts/{int(granularity*1000)} keV")
ax.set_xlabel(f"$E_x$ (MeV)")

if show_kinematic_line: 
    colors = cmap(np.linspace(0,1,len(residual_excitations)))
    for idx, ex in enumerate(residual_excitations):
        ax.axvline(ex,label=fr"$E_x$={ex:0.2f} MeV",color=colors[idx] )

ax.set_yscale("log")
ax.legend()
ax.set_ylim([0,25000])
fig.tight_layout()
fig.savefig("delete.png")
print(len(results["event"]))


In [ ]:
import polars as pl 
frame = {"run": results["run"][mask], "event": results["event"][mask]}
df = pl.DataFrame(frame)
# df.write_csv("./high_energy_deuterons.csv")